# 03 · Does correctly paired motion help?


We now hold the prediction problem fixed and change only the skeleton control.
The production experiment uses five source-disjoint outer folds and three
initialization seeds. Inner source folds choose hyperparameters. Outer test
sources never choose a ridge penalty, update count, or preferred arm.

The short example below fits one baseline and one two-update head on generated
data. It demonstrates the production APIs, not the complete scientific grid.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../docs/studies/future-innovation/README.md) ·
[Direct gate specification](../../../docs/studies/future-innovation/direct-gate-protocol.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation/direct-v2")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

## Execute this stage

Run the frozen direct-prediction comparison: five outer folds, three seeds, all frozen arms, and the complete frozen inner-selection grid. With FI_NOTEBOOK_FOLD set, run just that outer fold (all seeds and arms); HAIC supplies five CPU array tasks. Without it, run all folds sequentially. A verified audit rejection skips fitting and records a blocked outcome. No teaching settings enter this branch.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    fold = os.environ.get("FI_NOTEBOOK_FOLD")
    if fold is not None and fold not in {"0", "1", "2", "3", "4"}:
        raise ValueError("FI_NOTEBOOK_FOLD must be 0–4; unset it to run all five folds.")
    options = [] if fold is None else ["--outer-fold", fold]
    stage_error = attempt_stage("run-gate", RUN_ROOT, "--device", "cpu", *options)

## 1. Build every control inside its own partition

A mismatched donor must be from another source in the same training,
validation or test partition. Moving a donor across partitions changes
the information boundary. The production helper enforces different
sources; the nested runner supplies each partition separately.

Shuffle moves four-frame blocks with coordinates, confidence and
validity together. No-skeleton retains validity in the same-size head.
The real-minus-no-skeleton comparison measures what coordinate and
confidence history adds while matching the head's size and validity inputs.

In [ ]:
if MODE == "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import DIRECT_ARMS
    from gavd6_sjepa.research_directions.future_innovation.fi_controls import controlled_history
    rng = np.random.default_rng(260905)
    skeleton = rng.normal(size=(12, 32, 33, 4)).astype(np.float32)
    skeleton[..., 2] = 0.8  # confidence
    skeleton[..., 3] = 1.0  # validity
    source_ids = np.array([f"generated-source-{i // 2}" for i in range(12)])
    window_ids = np.array([f"generated-window-{i}" for i in range(12)])
    x = rng.normal(size=(12, 4))
    # This example is one training partition, never a mix of train and test.
    controls = {}
    for arm in DIRECT_ARMS:
        controls[arm], donors = controlled_history(arm, skeleton, window_ids, source_ids, x)
        if donors is not None:
            source_for = dict(zip(window_ids, source_ids))
            assert all(source_for[w] != source_for[d] for w, d in zip(window_ids, donors["donor_window_ids"]))
    assert np.array_equal(controls["no-skeleton"][..., 3], skeleton[..., 3])
    display(pd.DataFrame({"arm": list(controls), "history shape": [str(a.shape) for a in controls.values()]}))

## 2. Fit the baseline before defining its residual

Both scalers and the ridge fit see training examples only. Targets passed
to the residual head use the baseline's training-standardized units.
A lower training loss merely shows optimization on those examples.
It cannot establish held-out prediction quality.

This demonstration reduces width, target dimension and updates explicitly.
Real configuration remains owned by the initialized run; never transfer
these teaching settings into the frozen comparison.

In [ ]:
if MODE == "teach":
    import torch
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import ModelContract
    from gavd6_sjepa.research_directions.future_innovation.fi_residual_models import fit_baseline, train_head, predict_head
    y = x @ rng.normal(size=(4, 3)) + rng.normal(size=(12, 3)) * 0.2
    baseline = fit_baseline(x, y, window_ids, source_ids, alpha=1.0, variance_tolerance=1e-10)
    residual = baseline.y_scaler.transform(y) - baseline.predict(x)
    old_threads = torch.get_num_threads()
    try:
        torch.set_num_threads(1)  # Local teaching budget; restore before leaving.
        head, history = train_head(skeleton, baseline.x_scaler.transform(x), residual,
            source_ids, baseline.valid_features, seed=7, weight_decay=0.1,
            updates=(1, 2), model_contract=ModelContract(width=8, updates=(1, 2)), device="cpu")
        correction = predict_head(head, skeleton, baseline.x_scaler.transform(x), "cpu")
    finally:
        torch.set_num_threads(old_threads)
    assert correction.shape == y.shape and np.isfinite(correction).all()
    display(pd.DataFrame(history))
    print("Two CPU updates on generated training examples; no held-out scientific result.")

## 3. Inspect the real workload and reuse completed stages

Direct-v2 fits 5 outer folds × 3 seeds × 4 arms = 60 final heads,
plus training-only inner selection. Correct skeleton, time shuffle,
wrong-clip skeleton and a matched no-skeleton head share the same
person target and ridge baseline in each fold. Teacher features are
cached once, and CPU array tasks fit the heads. Legacy runs retain
their five-arm grid. The gate uses 50 clips; the full GAVD dataset is
reserved for the real experiment. No reduced teaching settings enter the fits.

Inspect saved configuration before submitting anything. Presence of a
fold receipt is not a fresh validation of its predictions. Existing
pipeline commands perform their own integrity and resumption checks.

In [ ]:
if MODE != "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_contracts import read_json
    config_path = RUN_ROOT / "config/model-contract.json"
    if config_path.is_file():
        display(read_json(config_path))
    else:
        print("No model configuration available locally; inspect the run's config directory on HAIC.")
    display(artifact_inventory(RUN_ROOT))
    audit_path = inspection_audit_path(RUN_ROOT)
    if audit_path.is_file():
        audit = read_json(audit_path)
        failed = [name for name, passed in audit.get("checks", {}).items() if passed is not True]
        if failed or audit.get("passed") is not True:
            print("FITTING BLOCKED by teacher validity:", ", ".join(failed) or "invalid summary")
            print("Inspect notebook 02 and its per-window QC. Do not tune thresholds to pass this run.")

## 4. Execute these notebooks on HAIC

Run these commands in a HAIC terminal only when you intend to execute the
real experiment. Set the paths to your checkout and existing run. For a
new run, first complete the environment/input setup in the
[HAIC guide](../../../slurm/future-innovation/README.md).

```bash
export GAVD6_ROOT=/path/to/gavd6
export FI_RUN_ROOT=/path/to/new/future-innovation/direct-v2
cd "$GAVD6_ROOT"
bash slurm/future-innovation/submit-fi-notebooks.sh all
```

`all` arranges preparation and compute dependencies and reuses verified
completed artifacts. Use `compute` when preparation has finished. The
run guide describes recovery; do not delete run contracts to resume.
The notebook itself never submits jobs. In `execute` mode its stage cell
launches the production fit grid, using the allocated CPU resources.

In [ ]:
if MODE == "execute":
    require_stage_success(stage_error)

## What this step establishes

Teaching mode illustrates a small fit. With passing validity audits, execute mode fits all folds or the selected array fold; a verified audit rejection records TRAINING BLOCKED without fitting. Scientific interpretation needs out-of-fold predictions for every arm and seed across all five folds. Next we score that complete comparison; job completion alone does not establish useful motion information.

Continue with [04_results_and_next_decision.ipynb](04_results_and_next_decision.ipynb).

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")